In [ ]:
from IPython.display import clear_output

%pip install kagglehub catboost xgboost tqdm imbalanced-learn -q

clear_output()


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

%matplotlib inline

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q1_data.csv")

df_del = pd.read_csv(csv_path) #df-del

In [ ]:
# Task 2: Write your code here:
df_del.head()

In [ ]:
# Task 3: Write your code here:
df_del.info()

In [ ]:
# Task 4: Write your code here:
df_del.describe()

In [ ]:
# Task 5: Write your code here:
condition_counts = df_del['Delivery_Time'].value_counts()
plt.figure(figsize=(10, 5))
plt.bar(condition_counts.index, condition_counts.values, color='coral')
plt.title('Condition Distribution')
plt.xlabel('Condition')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Task 1: Write your code here:
df_drop = df_del.drop(columns=['Order_ID'])
print(df_drop)

In [ ]:

# Task 2: Write your code here:
#print("\nMissing Values (df.isnull().sum()):")
#print(df_del.isnull().sum())

def check_missing_values(df):
  missing_values = df_del.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df_del)

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df_del.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_del)

In [ ]:


categorical_cols = df_del.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import LabelEncoder #import LabelEncoder

print('data before encoding:\n', categorical_cols) #show before encoding

label_encoder = LabelEncoder() # Instantiate LabelEncoder
for col in df_del.select_dtypes(include=["object"]).columns:
    df_del[col] = label_encoder.fit_transform(df_del[col])
data_label_encoded = label_encoder.fit_transform((categorical_cols).astype(str)) # Apply fit_transform to the column

print('\nData after encoding:\n', data_label_encoded) #show after encoding

In [ ]:
# Task 5: Write your code here:
# Random data generation for scaling
data_for_scale = pd.DataFrame({"Feature_1": [11, 40, 19,12],"Feature_2": [2384, 439, 3282,576]})

print("Before scaling:")
data_for_scale

from sklearn.preprocessing import StandardScaler

numerical_cols = df_del.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Task 6: Write your code here:
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df_del[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df_del, "Delivery_Time")


In [ ]:
# Task 1: Write your code here:

X = df_del.drop("Delivery_Time", axis=1).astype(float)
y = df_del['Delivery_Time'].astype(float)

In [ ]:
import numpy as np # for random data generation

np.random.seed(42) # for reproducbility

X_reg = pd.DataFrame({"Feature_1": np.random.randn(500)}) # features
y_reg = 3 * X_reg["Feature_1"] + np.random.randn(500) * 0.5  # continous labels with noise


X_clf = pd.DataFrame({"Feature_1": np.random.randn(500)}) # features
y_clf = (X_clf["Feature_1"] > 0).astype(int) # labels as class 0 or 1

In [ ]:
from sklearn.model_selection import train_test_split

# Use previously generated random data (example: classification data)
X, y = X_clf.copy(), y_clf.copy()

# split ratio (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 20% for test, remaining 80% for train
    random_state=42,      # reproducible output
    shuffle=True,         # representative splits
    stratify=y            # preserve class distribution
)
# Print shapes
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

In [ ]:
from sklearn.model_selection import StratifiedKFold

# Use previously generated random classification data
X, y = X_clf.copy(), y_clf.copy()

# Show full dataset class distribution
full_ratio = (y.value_counts(normalize=True) * 100).sort_index()
print("Full Dataset Class Distribution")
print("  y class percentages:", {k: f"{v:.2f}%" for k, v in full_ratio.items()})
print("-" * 40)

# Define Stratified K-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Stratified K-Fold Cross Validation\n" + "-"*40)

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
    # indexing for each fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    print(f"Fold {fold}")
    print("  X_train shape:", X_train.shape)
    print("  X_test shape :", X_test.shape)
    print("  y_train shape:", y_train.shape)
    print("  y_test shape :", y_test.shape)

    # showing class distribution
    train_ratio = (y_train.value_counts(normalize=True) * 100).sort_index()
    test_ratio = (y_test.value_counts(normalize=True) * 100).sort_index()

    print("  y_train class percentages:", {k: f"{v:.2f}%" for k, v in train_ratio.items()})
    print("  y_test class percentages :", {k: f"{v:.2f}%" for k, v in test_ratio.items()})

    print("-" * 40)






In [ ]:
from sklearn.ensemble import RandomForestRegressor

In [ ]:
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train_scaled, y_train)
print("Model trained!")

In [ ]:
from sklearn.metrics import mean_absolute_error

y_pred = model.predict(X_test_scaled)

mae = mean_absolute_error(y_test, y_pred)

print(f"MAE:  ${mae:,.2f}")



In [ ]:
# Task 1: Write your code here:

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: